In [ ]:
#セル1: 環境構築
# NumPy 2.0 非互換を回避
!pip install "numpy<2.0" --upgrade
#mecabライブラリの作成
!apt install -y libmecab-dev make mecab mecab-ipadic-utf8
#Pythonバインディングの導入
!pip install mecab-python3
#追加辞書IPADIC-NEologdの導入資源を取得
!git clone --depth 1 https://github.com/neologd/mecab-ipadic-neologd.git
#fileコマンドが無いため導入しておく
!sudo apt install file
#IPADIC-NEologd辞書の導入
!echo yes | mecab-ipadic-neologd/bin/install-mecab-ipadic-neologd -n -a
#runtimeエラーが起きるので、それに対する対処
# https://github.com/SamuraiT/mecab-python3#common-issues
!pip install unidic-lite
#fasttextの導入
#!pip install fasttext
!pip install fasttext==0.9.2
#表データ処理のpandasを導入
!pip install pandas
#科学計算ライブラリを導入
!pip install scikit-learn
#描画ライブラリの導入
!pip install matplotlib
#描画ライブラリの導入(グラフ)
!pip install seaborn


In [ ]:
#セル2: ライブラリのインポートと初期設定
import pandas as pd
import time
from datetime import datetime
import urllib.request
from bs4 import BeautifulSoup
from time import sleep
import fasttext as ft
import MeCab
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

#Matplotlib/Seabornにフォントを指定（インポート直後に1回だけ）
import matplotlib as mpl
from matplotlib import font_manager

# 保管先パスを定義
nlpDataPath = './nlpData/'
os.makedirs(nlpDataPath, exist_ok=True)

In [ ]:
#セル3: スクレイピング関数の定義
# [関数] 指定されたカテゴリのニュース一覧を取得し、タイトルをtitles設定する
def getNewsTitle(url, titles):
  # ニュース一覧のhtmlを取得
  html = urllib.request.urlopen(url)
  # htmlを分析しsoupオブジェクトを得る
  soup = BeautifulSoup(html, "html.parser")

  #<li>タグを探す、ニュースはこの中にある
  contents = soup.find_all("li")

  # 得られたliタグのコンテンツリストを順に処理
  for content in contents:
    #<h3>タグを探す、ニュースタイトルはこの中にある
    topic = content.find('h3')
    if (topic != None):
      #aタグを探す
      link = content.find('a')
      #リンクを取得(今回未使用となりますが)
      detailLink = link.get('href')
      #ニュースタイトルを取得し、titlesに設定
      titles.append(topic.text)

In [ ]:
#セル4: URLリストの定義
# ニュースを取り出すルートURL
# 最後のp=にページ番号を与えることで複数のページに切り替えてアクセスする。
# リストは順に、ラベル、ラベル名称(日本語)、ニュースカテゴリへのURL
urllist = [
    ['__label__0', '政治', 'https://news.livedoor.com/article/category/1/?p='],
    ['__label__1', '海外',  'https://news.livedoor.com/topics/category/world/?p='],
    ['__label__2', '映画', 'https://news.livedoor.com/article/category/52/?p='],
    ['__label__3',  'スポーツ', 'https://news.livedoor.com/topics/category/sports/?p=1'],
    ['__label__4',  '宇宙', 'https://news.livedoor.com/topics/keyword/32398/?p='],
]


In [ ]:
#セル5: スクレイピング実行
# 訓練データを格納するtrainData
trainData = []
# 検証データを格納するvalidationData
validationData=[]

# 分かち書きを行うため形態素解析器を準備
mc = MeCab.Tagger('-Owakati')

# 訓練データの作成
print('--- 訓練データの収集')
#ニュースカテゴリ数分実行
for u in urllist:
  titles = []
  # 1ページから順に5ページまでアクセス
  for i in range(5):
    # urlを組み立て、u[2]はullistの2番目の要素(ニュースサイトへのURL)
    # iをページ番号として利用するため、u[2]と結合する
    url = u[2]+str(i+1)
    # アクセスするURLを表示(進行条項確認のため)
    print(u[0], u[1], url)
    # ニュース取得関数を実行
    getNewsTitle(url, titles)
    # ニュースサイトに負荷をかけないためにスリープ
    sleep(0.5) #0.5秒sleep

  # 1ページにつき20件のニュースを取得。
  # ラベル(u[0])と取得したニュースタイトルを分かち書きして、trainDataに追加
  # 学習/検証データへの追加時はスペース区切り + strip で改行を除去
  for t in titles:
    wakati = mc.parse(t).strip()
    if wakati and len(wakati) > 5:  # 空行ガード + 最低5文字
      trainData.append(f'{u[0]} {wakati}')

#検証データの作成
print('--- 検証データの収集')
#ニュースカテゴリ数分実行
for u in urllist:
  titles = []
  url = u[2]+str(i+2)
  print(u[0], u[1], url)
  getNewsTitle(url, titles)
  sleep(0.5) #0.5秒sleep

  # 検証データ側も同様(改行削除・スペース区切り)
  for t in titles:
    wakati = mc.parse(t).strip()
    if wakati and len(wakati) > 5:
      validationData.append(f'{u[0]} {wakati}')

In [ ]:
#セル6: データの保存
#ダウンロードしたデータはファイルに保存しておく
#訓練データの保存
with open(nlpDataPath+'ftTrainData.txt', 'w') as f:
  f.write('\n'.join(trainData) + '\n')

#検証データの保存
with open(nlpDataPath+'ftValidationData.txt', 'w') as f:
  f.write('\n'.join(validationData) + '\n')

In [ ]:
#セル7: データのクリーニング（新規追加）
print("=== 学習データの確認 ===")
with open(nlpDataPath+'ftTrainData.txt', 'r') as f:
    lines = f.readlines()
    print(f"総行数: {len(lines)}")
    print(f"最初の5行:")
    for i, line in enumerate(lines[:5]):
        print(f"{i}: '{line.strip()}'")

    empty_lines = sum(1 for line in lines if len(line.strip()) < 10)
    print(f"短すぎる行(10文字未満): {empty_lines}")

def clean_training_data(input_file, output_file):
    """学習データをクリーニング"""
    cleaned_lines = []

    with open(input_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line and len(line) > 10 and '__label__' in line:
                parts = line.split(' ', 1)
                if len(parts) == 2 and parts[0].startswith('__label__') and len(parts[1].strip()) > 0:
                    cleaned_lines.append(line + '\n')

    with open(output_file, 'w') as f:
        f.writelines(cleaned_lines)

    print(f"クリーニング後: {len(cleaned_lines)}行")
    return len(cleaned_lines)

# データをクリーニング
train_count = clean_training_data(
    nlpDataPath+'ftTrainData.txt',
    nlpDataPath+'ftTrainData_cleaned.txt'
)
valid_count = clean_training_data(
    nlpDataPath+'ftValidationData.txt',
    nlpDataPath+'ftValidationData_cleaned.txt'
)

In [ ]:
#セル8: 学習（改善版）
# fasttextの学習(改善版)
if train_count > 0:
    try:
        model = ft.train_supervised(
            input=nlpDataPath+'ftTrainData_cleaned.txt',
            epoch=100,             # エポック数を増やす
            lr=0.1,                # 学習率を上げる
            wordNgrams=2,          # bi-gram使用
            loss="softmax",
            minCount=1,
            dim=200,               # 次元数を増やす
            ws=5,
            verbose=2
        )
        model.save_model(nlpDataPath+'fasttext.model')
        print("モデルの学習が完了しました")

        # 学習結果を確認
        N, P, R = model.test(nlpDataPath+'ftValidationData_cleaned.txt')
        print(f'\n【学習結果】')
        print(f'検証データ数: {N}')
        print(f'精度 (Precision): {P:.3f}')
        print(f'再現率 (Recall): {R:.3f}')

    except Exception as e:
        print(f"エラーが発生: {e}")
        print("\n代替案: より保守的なパラメータで再試行...")
        model = ft.train_supervised(
            input=nlpDataPath+'ftTrainData_cleaned.txt',
            epoch=25,
            lr=0.005,
            loss="softmax",
            minCount=2,
            dim=50,
            verbose=2
        )
        model.save_model(nlpDataPath+'fasttext.model')
        print("代替パラメータでモデル学習完了")
else:
    print("エラー: 有効な学習データがありません")

In [ ]:
#セル9: ラベル変換関数
# ラベル名を数値1桁に変換する関数
# 文字列ラベル名は扱いにくいため
# __label__0 ~ __label__4を順に0,1,2,3,4に変換
def mklabel(label):
    resp = 0

    if label == '__label__0':
        resp = 0
    elif label == '__label__1':
        resp = 1
    elif label == '__label__2':
        resp = 2
    elif label == '__label__3':
        resp = 3
    else:
        resp = 4

    return resp

In [ ]:
#セル10: 分類実行（修正版）
# 正解のラベルを格納(数値)
y_true = []
# 予測されたラベルを格納(数値)
y_pred = []

# 検証データをtに取得し分類を行う
# クリーニング済みファイルから読み込む
with open(nlpDataPath+'ftValidationData_cleaned.txt', 'r') as f:
    validationData = [line.strip() for line in f if line.strip()]

for t in validationData:
    # textに予測させるニュースタイトルを格納する
    parts = t.split(' ', 1)
    if len(parts) != 2:
        continue

    corr = parts[0]  # __label__0 などのラベル
    text = parts[1]  # 分かち書きされたテキスト

    # 予測させる(エラー回避版)
    try:
        predictions = model.predict(text, k=1)
        label = predictions[0]
        prob = predictions[1]

        # タプルやリストの場合の処理
        if isinstance(label, (tuple, list)):
            label = label[0] if len(label) > 0 else '__label__0'
        if isinstance(prob, (tuple, list)):
            prob = prob[0] if len(prob) > 0 else 0.0

    except Exception as e:
        print(f"予測エラー: {e}")
        label = '__label__0'
        prob = 0.0

    # 正解ラベルを数値に変換してy_trueへ格納
    y_true.append(mklabel(corr))

    # 予測されたラベルをクリーニング
    if isinstance(label, str):
        labelP = label
    else:
        labelP = str(label)

    labelP = labelP.replace(',', '').strip()

    # 予測された分類カテゴリを数値に変換
    y_pred.append(mklabel(labelP))

    # 結果表示
    print('正解={}, 予測={}, prob={:.3f}'.format(
        mklabel(corr), mklabel(labelP), float(prob) if prob else 0.0
    ))

print(f"\n処理完了: {len(y_true)}件")

In [ ]:
#セル11: 正解データの分布確認
# ここで正解ラベルの分布を確認します。
# 各ラベル20件ずつ取得していたので、
# グラフは同じになる事を確認します・
print('正解データ')
# 数値と分類クラスを表示
for u in urllist:
  print('{}={}'.format(mklabel(u[0]), u[1]))

# 描画ライブラリ(seaborn)を使ってグラフにプロットします
pdy_true = pd.DataFrame(y_true, columns=["class"])
sns.countplot(data=pdy_true, x="class")
plt.title('正解データの分布')
plt.xlabel('クラス')
plt.ylabel('件数')
plt.show()

In [ ]:
#セル12: 予測データの分布確認
# 予測された分類結果をグラフ表示し、
# 結果を俯瞰します。
print('予測データ')
# 数値と分類クラスを表示
for u in urllist:
  print('{}={}'.format(mklabel(u[0]), u[1]))

# 描画ライブラリ(seaborn)を使ってグラフにプロットします
pdy_pred = pd.DataFrame(y_pred, columns=["class"])
sns.countplot(data=pdy_pred, x="class")
plt.title('Distribution of Predicted Data') #予測データの分布
plt.xlabel('Class') #クラス
plt.ylabel('Number of items') #件数
plt.show()

In [ ]:
#セル13: 混同行列の表示
# 混同行列を表示し、分類精度を確認します
print('混同行列')
# 正解ラベルと予測した分類ラベルを指定して、混同行列を作成します。
cm = confusion_matrix(y_true, y_pred)
# 描画ライブラリ(seaborn)を使ってヒートマップをプロットします
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='d')
plt.title('confusion_matrix') #混同行列
plt.ylabel('Correct') #正解ラベル
plt.xlabel('Prediction') #予測ラベル
# プロットした混同行列を保存します
plt.savefig(nlpDataPath+'confusion_matrix.png')
plt.show()

# カテゴリ名を表示
print('\nカテゴリ対応:')
for u in urllist:
  print('{}={}'.format(mklabel(u[0]), u[1]))